# 🏙️ Paris Housing Prices — Notebook 01: Exploratory Data Analysis

Goal: understand the dataset structure, distributions, and key relationships before modelling.

**Dataset**: 1 200 residential properties across all 20 Paris arrondissements.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams["figure.dpi"] = 120

import warnings
warnings.filterwarnings("ignore")
print("Libraries loaded ✔")


## 1. Load & Inspect Data


In [ ]:
df = pd.read_csv("../data/paris_housing_prices_dataset.csv")
print(f"Shape: {df.shape}")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().round(2)


In [ ]:
print("Missing values:")
print(df.isnull().sum())
print()
print("Categorical value counts:")
for col in ["Property_Type", "Condition"]:
    print(col)
    print(df[col].value_counts())
    print()


## 2. Target Variable — Price Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["Price_EUR"], bins=40, color="#1a73e8", edgecolor="white", alpha=0.85)
axes[0].set_title("Price Distribution (€)", fontsize=14)
axes[0].set_xlabel("Price (€)")
axes[0].set_ylabel("Count")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{x/1e6:.1f}M"))

axes[1].hist(np.log1p(df["Price_EUR"]), bins=40, color="#1a73e8", edgecolor="white", alpha=0.85)
axes[1].set_title("log(Price + 1) Distribution", fontsize=14)
axes[1].set_xlabel("log(Price + 1)")

plt.tight_layout()
plt.savefig("../outputs/01_price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Mean price: €{df.Price_EUR.mean():,.0f}  |  Median: €{df.Price_EUR.median():,.0f}")


## 3. Price by Arrondissement


In [ ]:
arr_median = (df.groupby("Arrondissement")["Price_EUR"]
              .median()
              .sort_values(ascending=False)
              .reset_index())

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(arr_median["Arrondissement"].astype(str),
              arr_median["Price_EUR"] / 1e6,
              color="#1a73e8", edgecolor="white", alpha=0.85)
ax.set_title("Median Sale Price by Arrondissement", fontsize=15)
ax.set_xlabel("Arrondissement")
ax.set_ylabel("Median Price (€M)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{x:.1f}M"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/01_price_by_arrondissement.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Price by Property Type & Condition


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

order_type = df.groupby("Property_Type")["Price_EUR"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="Property_Type", y="Price_EUR", order=order_type,
            palette="Blues", ax=axes[0])
axes[0].set_title("Price by Property Type", fontsize=14)
axes[0].set_xlabel("")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{x/1e6:.1f}M"))

order_cond = df.groupby("Condition")["Price_EUR"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="Condition", y="Price_EUR", order=order_cond,
            palette="Blues", ax=axes[1])
axes[1].set_title("Price by Condition", fontsize=14)
axes[1].set_xlabel("")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{x/1e6:.1f}M"))

plt.tight_layout()
plt.savefig("../outputs/01_price_by_category.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Scatter: Price vs Numeric Features


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df["Size_sqm"], df["Price_EUR"] / 1e6,
                alpha=0.35, color="#1a73e8", s=18)
axes[0].set_title("Size (m²) vs Price", fontsize=14)
axes[0].set_xlabel("Size (m²)")
axes[0].set_ylabel("Price (€M)")

axes[1].scatter(df["Distance_to_Center_km"], df["Price_EUR"] / 1e6,
                alpha=0.35, color="#e84e0f", s=18)
axes[1].set_title("Distance to Centre vs Price", fontsize=14)
axes[1].set_xlabel("Distance to Centre (km)")
axes[1].set_ylabel("Price (€M)")

plt.tight_layout()
plt.savefig("../outputs/01_scatter_price.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Correlation Heatmap


In [ ]:
num_cols = ["Arrondissement","Size_sqm","Rooms","Floor",
            "Year_Built","Distance_to_Center_km","Price_EUR"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="Blues", linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig("../outputs/01_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Key Insights

| Observation | Detail |
|---|---|
| Price range | €100k – €5M+ with right-skewed distribution |
| Top arrondissements | 1st, 6th, 7th — highest median prices |
| Strongest correlator |  is the single strongest predictor |
| Distance effect | Prices decline as distance to centre increases |
| Condition premium | *New* and *Renovated* command a clear premium |

**→ Proceed to Notebook 02: Feature Engineering**
